Load dữ liệu

In [ ]:
import joblib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from google.colab import drive

# 1. Kết nối Drive & Load dữ liệu
try:
    drive.mount('/content/drive')
except:
    pass

save_dir = "/content/drive/MyDrive/DoAn_NIDS/Dataset/"

print("⏳ Đang load dữ liệu Train (đã cân bằng)...")
X_train = joblib.load(save_dir + 'X_train_final.pkl')
y_train = joblib.load(save_dir + 'y_train_final.pkl')

# Lấy số lượng đặc trưng đầu vào và số lượng lớp
input_dim = X_train.shape[1]
n_classes = len(np.unique(y_train))

print(f"✅ Input Dimension: {input_dim}")
print(f"✅ Number of Classes: {n_classes}")


In [ ]:
# 2. XÂY DỰNG MÔ HÌNH DNN
def build_dnn_model():
    model = Sequential(name="DNN_Model_Scenario_1")

    # --- Hidden Layer 1 ---
    model.add(Dense(512, input_dim=input_dim, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.2)) # Tắt ngẫu nhiên 20% nơ-ron để tránh học vẹt

    # --- Hidden Layer 2 ---
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    # --- Hidden Layer 3 ---
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    # --- Output Layer ---
    # Softmax: Chuyển đổi đầu ra thành xác suất (Tổng = 1)
    model.add(Dense(n_classes, activation='softmax'))

    # Compile mô hình
    optimizer = Adam(learning_rate=0.001)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])

    return model

# Khởi tạo và xem tóm tắt
model_dnn = build_dnn_model()
model_dnn.summary()

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# --- 1. CẤU HÌNH HUẤN LUYỆN ---
print("-" * 40)
print(f"🚀 Bắt đầu huấn luyện DNN (Kịch bản 1)...")

# Đường dẫn lưu model (File .keras là chuẩn mới, nhẹ và nhanh hơn .h5)
model_path = save_dir + 'Scenario1_DNN_Best.keras'

callbacks = [
    # 1. Lưu lại phiên bản tốt nhất (để dành báo cáo)
    ModelCheckpoint(model_path, monitor='val_loss', save_best_only=True, mode='min', verbose=1),

    # 2. Phanh gấp (Early Stopping): Quan trọng!
    # Nếu sau 5 vòng (patience=5) mà Loss không giảm thêm -> DỪNG NGAY.
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),

    # 3. Giảm tốc độ học nếu bị kẹt (Giúp leo xuống đáy thung lũng Loss sâu hơn)
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001, verbose=1)
]

# --- 2. THỰC HIỆN TRAINING ---
print("⏳ Đang chạy...")

history = model_dnn.fit(
    X_train, y_train,
    validation_split=0.1,  # 10% dữ liệu để chấm điểm nóng
    epochs=60,
    batch_size=256,       # Batch lớn giúp chạy nhanh trên tập dữ liệu triệu dòng
    callbacks=callbacks,
    verbose=1
)

print("\n✅ HUẤN LUYỆN HOÀN TẤT!")

# --- 3. VẼ BIỂU ĐỒ ĐÁNH GIÁ ---
plt.figure(figsize=(14, 5))

# Biểu đồ Loss (Càng thấp càng tốt)
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss', color='blue')
plt.plot(history.history['val_loss'], label='Val Loss', color='orange')
plt.title('Hàm mất mát (Loss) - Càng thấp càng tốt')
plt.xlabel('Epochs')
plt.legend()
plt.grid(True)

# Biểu đồ Accuracy (Càng cao càng tốt)
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='green')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='red')
plt.title('Độ chính xác (Accuracy) - Càng cao càng tốt')
plt.xlabel('Epochs')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import load_model

# 1. LOAD MÔ HÌNH TỐT NHẤT VÀ DỮ LIỆU TEST
print("⏳ Đang load lại mô hình DNN tốt nhất và dữ liệu Test...")
# Load model đã lưu lúc nãy
model_best = load_model(save_dir + 'Scenario1_DNN_Best.keras')

# Load tập Test (Đã chuẩn hóa) - Đảm bảo tính khách quan
X_test = joblib.load(save_dir + 'X_test_final.pkl')
y_test = joblib.load(save_dir + 'y_test_final.pkl')

# 2. THỰC HIỆN DỰ ĐOÁN
print("🚀 Đang chạy dự đoán trên tập Test (Vui lòng chờ)...")
y_pred_probs = model_best.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1) # Chuyển từ xác suất sang nhãn (0, 1, 2...)

# 3. TẠO BẢNG TÊN LỚP (Để hiển thị cho đẹp)
# (Thứ tự này phải khớp với lúc mã hóa Label Encoding ban đầu)
label_map = {i: label for i, label in enumerate(np.unique(y_test))}
target_names = [f"Class {i}" for i in range(len(label_map))] # Hoặc điền tên thật nếu anh có mapping

# 4. VẼ CONFUSION MATRIX
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(20, 15)) # Kích thước ảnh lớn để nhìn rõ 15 lớp
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=np.unique(y_test),
            yticklabels=np.unique(y_test))
plt.title('Confusion Matrix - DNN Model (Scenario 1)')
plt.ylabel('Nhãn thực tế (True Label)')
plt.xlabel('Nhãn dự đoán (Predicted Label)')
plt.show()

# 5. XUẤT BÁO CÁO CHI TIẾT (Precision, Recall, F1)
print("\n" + "="*60)
print("BẢNG ĐÁNH GIÁ CHI TIẾT (CLASSIFICATION REPORT)")
print("="*60)
print(classification_report(y_test, y_pred, digits=4))